# Quantum teleportation

For this Qiskit in Classrooms module, students must have a working Python environment with the following packages installed:
- `qiskit` v2.1.0 or newer
- `qiskit-ibm-runtime` v0.40.1 or newer
- `qiskit-aer` v0.17.0 or newer
- `qiskit.visualization`
- `numpy`
- `pylatexenc`

To set up and install the packages above, see the [Install Qiskit](https://quantum.cloud.ibm.com/docs/guides/install-qiskit) guide.
In order to run jobs on real quantum computers, students will need to set up an account with IBM Quantum&reg; by following the steps in the [Set up your IBM Cloud account](https://quantum.cloud.ibm.com/docs/guides/cloud-setup) guide.

This module was tested and used 14 seconds of QPU time. This is an estimate only. Your actual usage may vary.

In [ ]:
# Uncomment and modify this line as needed to install dependencies
#%pip install --upgrade "qiskit[visualization]" qiskit-ibm-runtime qiskit-aer numpy pylatexenc qc-grader

In [ ]:
from qc_grader.challenges.fallfest_2026 import check_progress

from qc_grader.challenges.fallfest_2026.teleport import (
    grade_teleport_ex1, grade_teleport_ex2, grade_teleport_ex3, grade_teleport_ex4,
    grade_teleport_ex5, grade_teleport_ex6, grade_teleport_ex7, grade_teleport_ex8, grade_teleport_ex9,
)

Watch the module walkthrough by Dr. Katie McCormick below, or click [here](https://youtu.be/jxqnzltpDdE?si=UVL58hFOOWe2Q9qI) to watch it on YouTube.

-------


<IBMVideo id="134413680" title="Katie McCormick demystifies quantum teleportation, and walks us through some examples using Qiskit."/>

## Introduction and background

Quantum teleportation transfers quantum information between qubits without physically moving the original particle. It uses entanglement, measurements, and classical communication. In this module, we will explore the theory and implement teleportation on a real quantum computer.

For more background, see John Watrous's [Basics of Quantum Information](https://quantum.cloud.ibm.com/learning/courses/basics-of-quantum-information), particularly the section on [Teleportation](https://quantum.cloud.ibm.com/learning/courses/basics-of-quantum-information/entanglement-in-action/quantum-teleportationr quantum teleportation.

We will be using the following scenario; Alice and Bob start in the same location and entangle their qubits. They will then move to two different locations, taking their respective qubits with them. Alice will then recieve a new qubit Q, which can be unknown to everybody, including Alice. Alice must transfer this information to Bob, using quantum teleportation. To do this we must use quantum operators, or 'gates'.

## Quantum operators (gates)

__H Hadamard Gate:__ Creates a superposition state. Single-qubit gate.
$$
H|0\rangle = \frac{1}{\sqrt{2}}\left(|0\rangle+|1\rangle\right),
$$

$$
H|1\rangle = \frac{1}{\sqrt{2}}\left(|0\rangle-|1\rangle\right)
$$

$$
H=\frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}
$$

A circuit with a Hadamard gate is made as follows:

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(1)
qc.h(0)
qc.draw("mpl")

__CNOT Controlled-NOT Gate:__ This gate uses two qubits: a control and a target. Checks the state of a control qubit which is not changed. But if the control qubit is in the state $|1\rangle$, the gate changes the state of the target qubit; if the state of the control qubit is $|0\rangle$ no change is made at all. In the notation below, assume the qubit $A$ (right-most qubit) is the control, and qubit $B$ (the left-most qubit) is the target. Below, the notation used is $CNOT(q_{control},q_{target})|BA\rangle.$ 

(Note: For those who have completed the Maths unit QIT, IBM uses the reverse CNOT rather than the one taught - this means the second qubit is the control, and the first is the target)

$$
CNOT(A,B)|00\rangle = |00\rangle, \\ CNOT(A,B)|01\rangle = |11\rangle, \\ CNOT(A,B)|10\rangle = |10\rangle, \\ CNOT(A,B)|11\rangle = |01\rangle
$$

You may sometimes see CNOT written with the order of the control and target simply implied. But there is no such ambiguity in code or in circuit diagrams.

$$
CNOT=\begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0\end{pmatrix}
$$

A CNOT gate looks a bit different in a circuit, since it requires two qubits. This is how it is implemented:

In [ ]:
qc = QuantumCircuit(2)
qc.cx(0, 1)
qc.draw("mpl")

**$X$ Gate**: Equivalent to a NOT operation. Single-qubit gate.

$$
X|0\rangle = |1\rangle,\\X|1\rangle=|0\rangle
$$

$$
X=\begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}
$$

In Qiskit, creating a circuit with an $X$ gate looks like this:

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)
qc.draw("mpl")

**$Z$ Gate**: Adds a "phase" to a state (a prefactor, which in the cases of the Z eigenstates $|0\rangle$ and $|1\rangle$ either a 1, or -1, respectively). Single-qubit gate.

$$
Z|0\rangle = |0\rangle,\\Z|1\rangle=-|1\rangle
$$

$$
Z=\begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}
$$

In Qiskit, creating a circuit with an $Z$ gate looks like this:

In [ ]:
qc = QuantumCircuit(1)
qc.z(0)
qc.draw("mpl")

## Theory

Let's describe quantum teleportation mathematically before implementing it on a quantum computer. 

**Alice and Bob entangle their qubits:** Initially, both qubits are in the state $|0\rangle$, giving the combined state $|00\rangle$.

Alice applies a Hadamard gate to her qubit, followed by a CNOT gate with Alice's qubit as the control and Bob's as the target:

$$
\begin{aligned}
CNOT(A,B)H_A|00\rangle
&=\frac{1}{\sqrt{2}}(|00\rangle+|11\rangle)
\end{aligned}
$$

The qubits are now **entangled**: their measurement outcomes are perfectly correlated, even though neither qubit has a definite individual state before measurement.

**Alice and Bob separate:** Alice and Bob move their entangled qubits to different locations. Moving quantum information can introduce errors, which we will explore in this module.

**Q is introduced:** Alice prepares the secret state on qubit Q:

$$
|\psi\rangle_Q=\alpha_0|0\rangle_Q+\alpha_1|1\rangle_Q
$$

The combined state of all three qubits is:

$$
\frac{1}{\sqrt{2}}(|00\rangle_{BA}+|11\rangle_{BA})
(\alpha_0|0\rangle_Q+\alpha_1|1\rangle_Q)
$$

Our goal is to transfer the quantum information encoded in Q to Bob's qubit using entanglement and classical communication.

__Alice entangles qubits A and Q:__ Alice now operates with a CNOT gate with her own qubit as the control and Q as the target, then applies a Hadamard gate to Q.  Let's calculate the three-qubit state after that operation:

$$
\begin{aligned}
H_Q CNOT(A,Q)|\psi\rangle_{AB}|\psi\rangle_Q &= H_Q CNOT(A,Q)\frac{1}{\sqrt{2}}\left(|0\rangle_B|0\rangle_A+|1\rangle_B|1\rangle_A\right)\left(\alpha_0 |0\rangle_Q + \alpha_1 |1\rangle_Q\right)\\
&= H_Q CNOT(A,Q)\frac{1}{\sqrt{2}}\left(\left(\alpha_0 |0\rangle_B|0\rangle_A|0\rangle_Q + \alpha_1 |0\rangle_B|0\rangle_A|1\rangle_Q\right)+\left(\alpha_0 |1\rangle_B|1\rangle_A|0\rangle_Q + \alpha_1 |1\rangle_B|1\rangle_A|1\rangle_Q\right)\right)\\
&= H_Q \frac{1}{\sqrt{2}}\left(\alpha_0 |0\rangle_B|0\rangle_A|0\rangle_Q + \alpha_1 |0\rangle_B|1\rangle_A|1\rangle_Q+\alpha_0 |1\rangle_B|1\rangle_A|0\rangle_Q + \alpha_1 |1\rangle_B|0\rangle_A|1\rangle_Q\right)\\
&= \frac{1}{2}\left(\alpha_0 |0\rangle_B|0\rangle_A|0\rangle_Q + \alpha_0 |0\rangle_B|0\rangle_A|1\rangle_Q + \alpha_1 |0\rangle_B|1\rangle_A|0\rangle_Q-\alpha_1 |0\rangle_B|1\rangle_A|1\rangle_Q\right)\\
&+\frac{1}{2}\left(\alpha_0 |1\rangle_B|1\rangle_A|0\rangle_Q + \alpha_0 |1\rangle_B|1\rangle_A|1\rangle_Q + \alpha_1 |1\rangle_B|0\rangle_A|0\rangle_Q - \alpha_1 |1\rangle_B|0\rangle_A|1\rangle_Q\right)
\end{aligned}
$$

Because A and Q are in the same location, let us group the terms above according to the outcomes of measurements on qubits A and Q:
$$
\begin{aligned}
|\psi\rangle = \frac{1}{2}\left((\alpha_0 |0\rangle_B+\alpha_1 |1\rangle_B)|0\rangle_A|0\rangle_Q +  (\alpha_0 |0\rangle_B-\alpha_1 |1\rangle_B)|0\rangle_A|1\rangle_Q + (\alpha_1 |0\rangle_B+\alpha_0 |1\rangle_B)|1\rangle_A|0\rangle_Q+ (-\alpha_1 |0\rangle_B+\alpha_0 |1\rangle_B)|1\rangle_A|1\rangle_Q \right)\\
\end{aligned}
$$

#### Check your understanding
Given the expression above for the states of all three qubits, what is the probability that a measurement of qubits A and Q yields $|0\rangle_A|0\rangle_Q?$

<details>
<summary>Answer</summary>

25%. To see this, recall that Bob's state must be normalized, so
$$ |_A \langle0|_Q\langle0| \frac{1}{2} |0\rangle_A|0\rangle_Q (\alpha_0 |0\rangle_B+\alpha_1 |1\rangle_B)|^2 = \frac{1}{4}|(\alpha_0 |0\rangle_B+\alpha_1 |1\rangle_B)|^2 = \frac{1}{4}$$

</details>

Alice measures qubits A and Q. There are four possible outcomes, each equally likely: $|0\rangle_A|0\rangle_Q,$ $|0\rangle_A|1\rangle_Q,$ $|1\rangle_A|0\rangle_Q,$ and $|1\rangle_A|1\rangle_Q.$

Each outcome determines the state of Bob's qubit. For example, if Alice measures \(A=0\) and \(Q=0\), Bob's qubit collapses to the original secret state:

$$
|\psi\rangle_B=\alpha_0|0\rangle_B+\alpha_1|1\rangle_B
$$

The other outcomes produce related states that Bob can recover using appropriate quantum gates, as shown in the table below.

| Alice outcome | Bob's state  | Instruction to Bob| Result |
| ----------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------- | -------------------------|
| $ \vert 0\rangle_A \vert 0\rangle_Q$ |     $\alpha_0\vert 0\rangle_B+\alpha_1\vert 1\rangle_B$        |        None           | $\alpha_0\vert 0\rangle_B+\alpha_1\vert 1\rangle_B$|
| $ \vert 0\rangle_A \vert 1\rangle_Q$ |     $\alpha_0\vert 0\rangle_B-\alpha_1\vert 1\rangle_B$        |        $Z$           |  $\alpha_0\vert 0\rangle_B+\alpha_1\vert 1\rangle_B$|
| $ \vert 1\rangle_A \vert 0\rangle_Q$ |     $\alpha_1\vert 0\rangle_B+\alpha_0\vert 1\rangle_B$       |        $X$           |  $\alpha_0\vert 0\rangle_B+\alpha_1\vert 1\rangle_B$|
| $ \vert 1\rangle_A \vert 1\rangle_Q$ |     $-\alpha_1\vert 0\rangle_B+\alpha_0\vert 1\rangle_B$       |        $X$ then $Z$           | $\alpha_0\vert 0\rangle_B+\alpha_1\vert 1\rangle_B$|

Bob's qubit is left in a state related to the secret state, but may have its coefficients swapped or a minus sign introduced. Alice sends Bob her measurement results through classical communication, telling him which gates to apply to recover the original state.

For example, an $X$ gate swaps the $|0\rangle$ and $|1\rangle$ components, correcting a state with swapped coefficients.

**Why can't this send information faster than light?** Bob cannot know which correction to apply until Alice communicates her measurement results. Classical communication limits the protocol to the speed of light.

On IBM&reg; quantum computers, qubits are fixed on a chip rather than physically separated. We can use SWAP gates to move quantum information between qubits, simulating the separation of Alice and Bob.

## Experiment 1: Basic teleportation

IBM Quantum recommends tackling quantum computing problems using a framework we call "Qiskit patterns". It consists of the following steps.
- Step 1: Map your problem to a quantum circuit
- Step 2: Optimize your circuit for running on real quantum hardware
- Step 3: Execute your job on IBM quantum computers using Runtime Primitives
- Step 4: Post-process the results

### Step 1: Map your problem to a quantum circuit

All the math we did above was outlining step 1. We will implement it now, building our quantum circuit using Qiskit! We start creating a quantum circuit with three qubits, and entangling the two qubits of Alice and Bob. We will take these to be qubits 1 and 2, and we will reserve qubit 0 for the secret state.

In [ ]:
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
import numpy as np

#Define registers---------------------------------------------------------------------------------------------------------------------------------------

secret = QuantumRegister(1, "Q")
Alice = QuantumRegister(1, "A")
Bob = QuantumRegister(1, "B")

cr = ClassicalRegister(3, "c")

qc = QuantumCircuit(secret, Alice, Bob, cr)

#------------------------------------------------------------------------------------------------------------------------------------------------------

# We entangle Alice's and Bob's qubits as in our work above.
# Apply a Hadamard gate to Alice's qubit, then a CNOT gate
# with Alice as the control and Bob as the target.



# YOUR CODE HERE


#-------------------------------------------------------------------------------------------------------------------------------------------------------

# Inserting a barrier changes nothing about the logic. It just allows us to
# force gates to be positioned in "layers".

qc.barrier()

#-------------------------------------------------------------------------------------------------------------------------------------------------------

# Now we will use random variables to create the secret state.
# Don't worry about the "u" gate and the details.

np.random.seed(42)  # fixing seed for repeatability
theta = np.random.uniform(0.0, 1.0) * np.pi
varphi = np.random.uniform(0.0, 2.0) * np.pi

#-------------------------------------------------------------------------------------------------------------------------------------------------------

# Assign the secret state to the qubit on the other side of Alice's (qubit 0), labeled Q.
# Use the general single-qubit unitary gate to prepare the state.

# YOUR CODE HERE



qc.barrier()
#-------------------------------------------------------------------------------------------------------------------------------------------------------

# Now entangle Q and Alice's qubits as in the discussion above.
# Apply a CNOT gate with secret as the control and Alice as the target.
# Then apply a Hadamard gate to secret.

# YOUR CODE HERE




qc.barrier()
#-------------------------------------------------------------------------------------------------------------------------------------------------------

# Now Alice measures her qubits, and stores the outcomes in the "classical registers" cr[]
qc.measure(Alice, cr[1])
qc.measure(secret, cr[0])

#-------------------------------------------------------------------------------------------------------------------------------------------------------

# Now we insert some conditional logic. If Alice measures Q in a "1" we need a Z gate, and if Alice
# measures A in a "1" we need an X gate (see the table).

# If Alice's measurement of A (stored in cr[1]) is 1, apply an X gate to Bob.

# YOUR CODE HERE



# If Alice's measurement of Q (stored in cr[0]) is 1, apply a Z gate to Bob.

# YOUR CODE HERE



qc.draw(output="mpl")

After applying the corrections, Bob has the secret state. However, measuring a qubit only gives $|0\rangle$ or $|1\rangle$, so we cannot directly verify an arbitrary quantum state.

To check the teleportation worked, we apply the inverse of the unitary gate $U$ used to prepare the secret state. This returns the state to $|0\rangle$, which we can measure. This step is only for verification, not required for teleportation itself.

In [ ]:
# Add a barrier to separate the final operations from the previous gates.
qc.barrier()

#-------------------------------------------------------------------------------------------------------------------------------------------------------
# Apply the inverse of the U gate to Bob's qubit to recover |0⟩.

# YOUR CODE HERE


#------------------------------------------------------------------------------------------------------------------------------------------------------
# Measure Bob's qubit and store the result in cr[2].


# YOUR CODE HERE

#-------------------------------------------------------------------------------------------------------------------------------------------------------
qc.draw(output="mpl")

If teleportation worked correctly, measuring Bob's qubit after applying the inverse of $U$ should give $|0\rangle$. However, measurements are probabilistic, so we need to repeat the experiment many times to check that the probability of measuring $|0\rangle$ is high.

### Step 2: Optimize for quantum execution

Next, we prepare the circuit for a specific quantum computer. This involves selecting a backend and adapting the circuit to the device's available gates and qubit layout.

We will load the required packages and choose a backend, either the least busy available option or a specific device.

There is code below for saving your credentials upon first use. Be sure to delete this information from the notebook after saving it to your environment, so that your credentials are not accidentally shared when you share the notebook. See [Set up your IBM Cloud account](https://quantum.cloud.ibm.com/docs/guides/initialize-account) and [Initialize the service in an untrusted environment](https://quantum.cloud.ibm.com/docs/guides/cloud-setup-untrusted) for more guidance.

In [ ]:
# Load the Qiskit Runtime service
from qiskit_ibm_runtime import QiskitRuntimeService

# Load the Qiskit Runtime service

# Syntax for first saving your token.  Delete these lines after saving your credentials.
# QiskitRuntimeService.save_account(channel='ibm_quantum_platform',
# instance = '<YOUR_IBM_INSTANCE_CRN>', token='<YOUR-API_KEY>', overwrite=True, set_as_default=True)
# service = QiskitRuntimeService(channel='ibm_quantum_platform')

# Load saved credentials
service = QiskitRuntimeService()

# Use the least busy backend, or uncomment the loading of a specific backend like "ibm_brisbane".
backend = service.least_busy(operational=True)
# backend = service.backend("ibm_brisbane")
print(backend.name)

We explicitly enable logic on measurements.

In [ ]:
from qiskit.circuit import IfElseOp

backend.target.add_instruction(IfElseOp, name="if_else")

Next, we **transpile** the circuit, converting its gates into instructions supported by the chosen quantum computer and adapting it to the device's qubit layout.

We use a preset pass manager with `optimization_level=3`, the highest optimization level, to simplify the circuit and improve its execution efficiency.

In [ ]:
# Step 2: Transpile
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
qc_isa = pm.run(qc)

A "Sampler" is a primitive designed to sample possible states resulting from a quantum circuit, and collect statistics on what states might be measured and with what probability. We import the Qiskit Runtime Sampler here:

In [ ]:
# Load the Runtime primitive and session
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

Although this teleportation circuit can be simulated on a classical computer, we recommend running it on a real IBM quantum computer to experience quantum hardware.

If hardware access is unavailable or time is limited, you can use a simulator instead. Run the cell below and uncomment the corresponding lines in the **Execute** steps.

In [ ]:
# Load the backend sampler
from qiskit.primitives import BackendSamplerV2

# Load the Aer simulator and generate a noise model based on the currently-selected backend.
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel


noise_model = NoiseModel.from_backend(backend)

# Define a simulator using Aer, and use it in Sampler.
backend_sim = AerSimulator(noise_model=noise_model)
sampler_sim = BackendSamplerV2(backend=backend_sim)

# Alternatively, load a fake backend with generic properties and define a simulator.
# backend_gen = GenericBackendV2(num_qubits=18)
# sampler_gen = BackendSamplerV2(backend=backend_gen)

### Step 3: Execute

Use the sampler to run your job, with the circuit as an argument.

In [ ]:
job = sampler.run([qc_isa])
# job = sampler_sim.run([qc_isa])
res = job.result()
counts = res[0].data.c.get_counts()

### Step 4: Post-processing and analysis

Let's plot the results and interpret them.

In [ ]:
# This required 5 s to run on a Heron r2 processor on 10-28-24
from qiskit.visualization import plot_histogram

plot_histogram(counts)

#### Check your understanding
Which of the states above indicate successful teleportation, and how can you tell?

<details>
<summary>Answer</summary>

The states $|000\rangle,$ $|001\rangle,$ $|010\rangle,$ $|011\rangle$ are all consistent with successful teleportation. This is because we added a gate to undo the initial preparation of the secret state. If the secret state was successfully teleported to Bob's qubit, that additional gate should return Bob's qubit to the $|0\rangle$ state. So any state above with Bob's qubit (qubit 0, also measured to the 0th component of the classical register, and hence the highest/right-most) in the $|0\rangle$ state indicates success.

  </details>

The plot shows measurement outcomes across 5,000 trials. Alice's two measurement results should occur with roughly equal probabilities.

The qubits are ordered Q, A, B in the circuit. The four outcomes with Bob's qubit in $|0\rangle$ should account for most measurements, indicating successful teleportation.

A small number of results may show Bob's qubit in $|1\rangle$ due to noise and hardware errors. Real quantum computers are imperfect, and quantum error correction remains an important area of research.

## Experiment 2: Teleporting across a processor

In this experiment, we will teleport quantum information between qubits positioned far apart on a quantum processor. Since physical qubits cannot be moved, we use SWAP gates to transfer information between qubits across the chip.

### Step 1: Map your problem to a quantum circuit

Unlike the previous experiment, Alice's and Bob's qubits will change as the information moves. We will number the qubits and use variables to track their current positions. The teleportation steps remain the same, with additional SWAP gates to move the information across the processor.

In [ ]:
# Step 1: Map

# Define registers-------------------------------------------------------------------------------------------------------------------------------------

qr = QuantumRegister(13, "q")

qc = QuantumCircuit(qr, cr)

# Define registers--------------------------------------------------------------------------------------------------------------------------------------

secret = QuantumRegister(1, "Q")
ebitsa = QuantumRegister(6, "A")
ebitsb = QuantumRegister(6, "B")

cr = ClassicalRegister(3, "c")
qc = QuantumCircuit(secret, ebitsa, ebitsb, cr)

# We'll start Alice in the middle of the circuit, then move information outward in both directions.-----------------------------------------------------

Alice = 5
Bob = 0

# Apply a Hadamard gate to Alice's qubit and a CNOT gate to entangle Alice and Bob.---------------------------------------------------------------------

# YOUR CODE HERE



# Starting with Bob and Alice in the center, we swap their information onto adjacent qubits,------------------------------------------------------------
# until the information is on distant qubits.

for n in range(Alice):
    # Swap Bob's information with the adjacent qubit.
    
    # YOUR CODE HERE

    # Swap Alice's information with the adjacent qubit.
    
    # YOUR CODE HERE

    Alice = Alice - 1
    Bob = Bob + 1

qc.barrier()

# Create a random state for Alice (qubit zero)----------------------------------------------------------------------------------------------------------

np.random.seed(42)  # fixing seed for repeatability
# theta = np.random.uniform(0.0, 1.0) * np.pi
theta = 0.3
varphi = np.random.uniform(0.0, 2.0) * np.pi

# Prepare the secret state on Q using the U gate.-------------------------------------------------------------------------------------------------------

# YOUR CODE HERE


# Entangle Alice's two qubits---------------------------------------------------------------------------------------------------------------------------
# Apply a CNOT gate, then a Hadamard gate.

# YOUR CODE HERE

qc.barrier()

# Make measurements of Alice's qubits and store the results in the classical register.------------------------------------------------------------------
qc.measure(ebitsa[Alice], cr[1])
qc.measure(secret, cr[0])

# Send instructions to Bob's qubits based on the outcome of Alice's measurements.-----------------------------------------------------------------------
# If cr[1] is 1, apply an X gate to Bob.

# YOUR CODE HERE

# If cr[0] is 1, apply a Z gate to Bob.

# YOUR CODE HERE

qc.barrier()

# Invert the preparation we did for Bob's qubit so we can check whether we did this correctly.----------------------------------------------------------

# YOUR CODE HERE

# Measure Bob's qubit and store the result in cr[2].----------------------------------------------------------------------------------------------------

# YOUR CODE HERE

qc.draw("mpl")

The circuit follows the same teleportation steps as before, but uses SWAP gates to move Alice's information from qubit 6 ($A_5$) to qubit 1 ($A_0$), next to Q. Bob's information is moved from qubit 7 ($B_0$) to qubit 12 ($B_5$).

Bob's qubit only recovers the secret state after Alice measures her qubits and Bob applies the required conditional $X$ and $Z$ gates.

### Step 2: Optimize your circuit

Normally, we use `optimization_level=3` to optimize circuits. However, here the SWAP gates demonstrate long-distance teleportation. Optimization could remove these unnecessary swaps, so we use `optimization_level=0` to preserve the intended circuit structure.

In [ ]:
# Step 2: Transpile
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

target = backend.target
pmzero = generate_preset_pass_manager(target=target, optimization_level=0)

qc_isa_zero = pmzero.run(qc)

print(qc_isa_zero.depth())

We can visualize where on the quantum processor these qubits are using the ```plot_circuit_layout``` function.

### Step 3: Execute

As before, we recommend running on real IBM quantum computers. If your monthly free usage has been reached, feel free to uncomment the simulator cells to run on a simulator.

In [ ]:
# This required 5 s to run on a Heron r2 processor on 10-28-24
job = sampler.run([qc_isa_zero])
# job = sampler_sim.run([qc_isa_zero])
counts = job.result()[0].data.c.get_counts()

In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(counts)

### Step 4: Classical post-processing

Again we see that the probabilities for the possible outcomes for Alice's qubits are fairly uniform. There is a strong preference for finding Bob's qubit in $|0\rangle$ after inverting the secret code, meaning there is a strong probability that we correctly teleported the secret state across the processor from Q to Bob (qubits 0 to 12). However, we note that there is now about a higher chance of *not* measuring $|0\rangle$ for Bob. This is an important lesson in quantum computing: the more gates you have, especially multi-qubit gates like swap gates, the more noise and errors you will encounter.

## Questions

### Critical concepts

- Qubits can be entangled, meaning a measurement of one qubit affects or even determines the state of another qubit.
- Through a combination of entanglements and measurements, we can transfer a state from one qubit to another — even over long distances. This is called quantum teleportation.
- Quantum teleportation relies on quantum measurements, which are probabilistic. Classical communication is necessary to complete the transfer. This prevents quantum teleportation from moving information faster than light.
- Modern quantum computers are more susceptible to noise and errors than classical computers.

### True/False Questions

**1.** Quantum teleportation can be used to send information faster than light.

In [ ]:
answer = True  # Replace with True or False
grade_teleport_ex1(answer)

**2.** Modern evidence suggests that the collapse of a quantum state propagates faster than light.

In [ ]:
answer = True  # Replace with True or False
grade_teleport_ex2(answer)

**3.** In Qiskit, qubits are ordered in states with the lowest-numbered qubit on the right, as in $|q_3,q_2,q_1, q_0\rangle$.

In [ ]:
answer = True  # Replace with True or False
grade_teleport_ex3(answer)

### Multiple Choice Questions

**1.** Qubits A and B are entangled, then separated by a great distance $d$. Qubit A is measured. Which statement is correct about the speed at which the state of qubit B is affected?

- a. Qubit B is affected instantly, within experimental tolerance.
- b. Qubit B is affected after time $d/c$ (speed of light).
- c. Qubit B is affected only after classical communication has occurred.
- d. None of the above.

In [ ]:
answer = "a"  # Replace with your answer: "a", "b", "c", or "d"
grade_teleport_ex4(answer)

**2.** The measurement histogram shown could correspond to which of the following quantum states?

<img src="https://quantum.cloud.ibm.com/learning/images/modules/computer-science/quantum-teleportation/entangled_teleportation_fig.avif" alt="entangled_teleportation_fig" width="400">

*entangled_teleportation_fig*

- a. $|0\rangle$
- b. $\frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$
- c. $\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$
- d. $\frac{4}{5}|0\rangle+\frac{3}{5}|1\rangle$
- e. $\frac{3}{5}|0\rangle+\frac{4}{5}|1\rangle$

In [ ]:
answer = "a"  # Replace with your answer: "a", "b", "c", "d", or "e"
grade_teleport_ex5(answer)

**3.** Which of the following states show qubits A and B entangled? Select all that apply.

- a. $\frac{1}{\sqrt{2}}(|0\rangle_B|0\rangle_A+|1\rangle_B|1\rangle_A)$
- b. $\frac{4}{5}|0\rangle_B|0\rangle_A+\frac{3}{5}|1\rangle_B|1\rangle_A$
- c. $\frac{1}{\sqrt{2}}(|0\rangle_B|1\rangle_A-|1\rangle_B|0\rangle_A)$
- d. $\frac{1}{\sqrt{2}}(|0\rangle_B|0\rangle_A+|1\rangle_B|0\rangle_A)$
- e. $|0\rangle_B|0\rangle_A$

In [ ]:
answer = ["a", "b"]  # Replace with a list of your answers, e.g. ["a", "c"]
grade_teleport_ex6(answer)

**4.** Which state could yield a 2-qubit measurement histogram with peaks at $|01\rangle$ and $|10\rangle$?

<img src="https://quantum.cloud.ibm.com/learning/images/modules/computer-science/quantum-teleportation/entangled_teleportation_fig_0110.avif" alt="entangled_teleportation_fig_0110" width="400">

*entangled_teleportation_fig_0110*

- a. $\frac{1}{\sqrt{2}}(|0\rangle_B|0\rangle_A+|1\rangle_B|1\rangle_A)$
- b. $\frac{4}{5}|0\rangle_B|0\rangle_A+\frac{3}{5}|1\rangle_B|1\rangle_A$
- c. $\frac{1}{\sqrt{2}}(|0\rangle_B|1\rangle_A-|1\rangle_B|0\rangle_A)$
- d. $\frac{4}{5}|0\rangle_B|1\rangle_A+\frac{3}{5}|1\rangle_B|0\rangle_A$
- e. $|0\rangle_B|0\rangle_A$

In [ ]:
answer = "a"  # Replace with your answer: "a", "b", "c", "d", or "e"
grade_teleport_ex7(answer)

### Discussion Questions

**1.** Describe the quantum teleportation protocol, from start to finish, to your partner/group.

In [ ]:
answer = """Type your answer here"""
grade_teleport_ex8(answer)

**2.** Is there anything unique about the initial entangled state $\frac{1}{\sqrt{2}}(|0\rangle_B|0\rangle_A+|1\rangle_B|1\rangle_A)$? What other entangled states could have been used?

In [ ]:
answer = """Type your answer here"""
grade_teleport_ex9(answer)

In [ ]:
check_progress("teleport")